In [2]:
import mysql.connector
from mysql.connector import Error

def test_connection():
    connection = None
    try:
        # Replace with your actual credentials
        connection = mysql.connector.connect(
            host='localhost',        # Your MySQL host
            database='stu_score', # Your database name
            user='root',     # Your MySQL username
            password='', # Your MySQL password
            port=3306                 # Default MySQL port
        )
        
        if connection.is_connected():
            db_info = connection.get_server_info()
            print(f"Successfully connected to MySQL Server version {db_info}")
            
            cursor = connection.cursor()
            cursor.execute("SELECT DATABASE();")
            record = cursor.fetchone()
            print(f"Connected to database: {record[0]}")
            
    except Error as e:
        print(f"Error while connecting to MySQL: {e}")
    finally:
        if connection and connection.is_connected():
            cursor.close()
            connection.close()
            print("MySQL connection closed")

if __name__ == "__main__":
    test_connection()

Successfully connected to MySQL Server version 9.6.0
Connected to database: stu_score
MySQL connection closed


/var/folders/pz/1xf6kv15419_mj8wbrhzqlcw0000gp/T/ipykernel_84196/1713183009.py:17: DeprecationWarning: Call to deprecated function get_server_info. Reason: 
    The property counterpart 'server_info' should be used instead.

  db_info = connection.get_server_info()


## create a new stu info

In [4]:
import mysql.connector
from mysql.connector import Error

def create_student_table():
    connection = None
    cursor = None
    try:
        # Establish connection (update with your credentials)
        connection = mysql.connector.connect(
            host='localhost',
            database='stu_score',  # Replace with your database name
            user='root',       # Replace with your username
            password=''    # Replace with your password
        )
        
        if connection.is_connected():
            cursor = connection.cursor()
            
            # SQL query to create student table
            create_table_query = """
            CREATE TABLE IF NOT EXISTS students (
                student_id INT AUTO_INCREMENT PRIMARY KEY,
                first_name VARCHAR(50) NOT NULL,
                last_name VARCHAR(50) NOT NULL,
                date_of_birth DATE,
                email VARCHAR(100) UNIQUE NOT NULL,
                phone VARCHAR(20),
                address TEXT,
                city VARCHAR(50),
                state VARCHAR(50),
                country VARCHAR(50),
                postal_code VARCHAR(20),
                enrollment_date DATE DEFAULT (CURRENT_DATE),
                student_status ENUM('active', 'inactive', 'graduated', 'suspended') DEFAULT 'active',
                gpa DECIMAL(3,2) CHECK (gpa >= 0 AND gpa <= 4.0),
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP
            ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;
            """
            
            cursor.execute(create_table_query)
            connection.commit()
            print("Student table 'students' created successfully!")
            
    except Error as e:
        print(f"Error: {e}")
    finally:
        if cursor:
            cursor.close()
        if connection and connection.is_connected():
            connection.close()
            print("MySQL connection closed")

if __name__ == "__main__":
    create_student_table()

Student table 'students' created successfully!
MySQL connection closed


In [6]:
import mysql.connector
from mysql.connector import Error
from datetime import datetime
import re

class StudentDatabase:
    def __init__(self, host, database, user, password):
        self.host = host
        self.database = database
        self.user = user
        self.password = password
        self.connection = None
        self.cursor = None
    
    def connect(self):
        """Establish database connection"""
        try:
            self.connection = mysql.connector.connect(
                host=self.host,
                database=self.database,
                user=self.user,
                password=self.password
            )
            
            if self.connection.is_connected():
                self.cursor = self.connection.cursor(dictionary=True)
                print("✅ Connected to MySQL database")
                return True
        except Error as e:
            print(f"❌ Connection error: {e}")
            return False
    
    def create_student_table(self):
        """Create students table with constraints"""
        create_table_query = """
        CREATE TABLE IF NOT EXISTS students (
            student_id INT AUTO_INCREMENT PRIMARY KEY,
            first_name VARCHAR(50) NOT NULL,
            last_name VARCHAR(50) NOT NULL,
            date_of_birth DATE NOT NULL,
            email VARCHAR(100) UNIQUE NOT NULL,
            phone VARCHAR(20),
            address TEXT,
            city VARCHAR(50),
            state VARCHAR(50),
            country VARCHAR(50) DEFAULT 'USA',
            postal_code VARCHAR(20),
            enrollment_date DATE DEFAULT (CURRENT_DATE),
            student_status ENUM('active', 'inactive', 'graduated', 'suspended') DEFAULT 'active',
            gpa DECIMAL(3,2) DEFAULT 0.00 CHECK (gpa >= 0 AND gpa <= 4.0),
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
            
            -- Add indexes for better performance
            INDEX idx_email (email),
            INDEX idx_status (student_status),
            INDEX idx_enrollment (enrollment_date)
            
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;
        """
        
        try:
            self.cursor.execute(create_table_query)
            self.connection.commit()
            print("✅ Table 'students' created successfully!")
            return True
        except Error as e:
            print(f"❌ Error creating table: {e}")
            return False
    
    def add_student(self, student_data):
        """Add a new student to the database"""
        insert_query = """
        INSERT INTO students 
        (first_name, last_name, date_of_birth, email, phone, address, 
         city, state, country, postal_code, enrollment_date, gpa)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """
        
        try:
            self.cursor.execute(insert_query, student_data)
            self.connection.commit()
            print(f"✅ Student {student_data[0]} {student_data[1]} added successfully!")
            return self.cursor.lastrowid
        except Error as e:
            print(f"❌ Error adding student: {e}")
            return None
    
    def get_all_students(self):
        """Retrieve all students"""
        query = """
        SELECT student_id, first_name, last_name, email, 
               date_of_birth, enrollment_date, student_status, gpa
        FROM students
        ORDER BY last_name, first_name
        """
        
        try:
            self.cursor.execute(query)
            students = self.cursor.fetchall()
            return students
        except Error as e:
            print(f"❌ Error fetching students: {e}")
            return []
    
    def get_student_by_id(self, student_id):
        """Retrieve a specific student by ID"""
        query = "SELECT * FROM students WHERE student_id = %s"
        
        try:
            self.cursor.execute(query, (student_id,))
            student = self.cursor.fetchone()
            return student
        except Error as e:
            print(f"❌ Error fetching student: {e}")
            return None
    
    def update_student(self, student_id, update_data):
        """Update student information"""
        update_query = """
        UPDATE students 
        SET first_name = %s, last_name = %s, phone = %s, address = %s,
            city = %s, state = %s, postal_code = %s, student_status = %s
        WHERE student_id = %s
        """
        
        try:
            self.cursor.execute(update_query, (*update_data, student_id))
            self.connection.commit()
            print(f"✅ Student ID {student_id} updated successfully!")
            return True
        except Error as e:
            print(f"❌ Error updating student: {e}")
            return False
    
    def update_gpa(self, student_id, new_gpa):
        """Update student's GPA"""
        if 0 <= new_gpa <= 4.0:
            query = "UPDATE students SET gpa = %s WHERE student_id = %s"
            try:
                self.cursor.execute(query, (new_gpa, student_id))
                self.connection.commit()
                print(f"✅ GPA updated for student ID {student_id}")
                return True
            except Error as e:
                print(f"❌ Error updating GPA: {e}")
                return False
        else:
            print("❌ GPA must be between 0.0 and 4.0")
            return False
    
    def delete_student(self, student_id):
        """Delete a student from the database"""
        query = "DELETE FROM students WHERE student_id = %s"
        
        try:
            self.cursor.execute(query, (student_id,))
            self.connection.commit()
            if self.cursor.rowcount > 0:
                print(f"✅ Student ID {student_id} deleted successfully!")
                return True
            else:
                print(f"❌ Student ID {student_id} not found")
                return False
        except Error as e:
            print(f"❌ Error deleting student: {e}")
            return False
    
    def search_students(self, search_term):
        """Search students by name or email"""
        query = """
        SELECT student_id, first_name, last_name, email, student_status, gpa
        FROM students 
        WHERE first_name LIKE %s OR last_name LIKE %s OR email LIKE %s
        ORDER BY last_name, first_name
        """
        search_pattern = f"%{search_term}%"
        
        try:
            self.cursor.execute(query, (search_pattern, search_pattern, search_pattern))
            results = self.cursor.fetchall()
            return results
        except Error as e:
            print(f"❌ Error searching students: {e}")
            return []
    
    def get_statistics(self):
        """Get student statistics"""
        query = """
        SELECT 
            COUNT(*) as total_students,
            AVG(gpa) as average_gpa,
            MIN(gpa) as min_gpa,
            MAX(gpa) as max_gpa,
            SUM(CASE WHEN student_status = 'active' THEN 1 ELSE 0 END) as active_students,
            SUM(CASE WHEN student_status = 'graduated' THEN 1 ELSE 0 END) as graduated_students
        FROM students
        """
        
        try:
            self.cursor.execute(query)
            stats = self.cursor.fetchone()
            return stats
        except Error as e:
            print(f"❌ Error getting statistics: {e}")
            return None
    
    def close(self):
        """Close database connection"""
        if self.cursor:
            self.cursor.close()
        if self.connection and self.connection.is_connected():
            self.connection.close()
            print("🔒 Database connection closed")

def validate_email(email):
    """Validate email format"""
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return re.match(pattern, email) is not None

def main():
    # Initialize database connection
    db = StudentDatabase(
        host='localhost',
        database='stu_score',
        user='root',
        password=''
    )
    
    # Connect to database
    if not db.connect():
        return
    
    # Create table
    db.create_student_table()
    
    # Add sample students
    students_data = [
        ('John', 'Doe', '2000-05-15', 'john.doe@email.com', '555-0101', 
         '123 Main St', 'New York', 'NY', 'USA', '10001', '2023-09-01', 3.5),
        ('Jane', 'Smith', '2001-08-22', 'jane.smith@email.com', '555-0102',
         '456 Oak Ave', 'Los Angeles', 'CA', 'USA', '90001', '2023-09-01', 3.8),
        ('Mike', 'Johnson', '1999-12-10', 'mike.j@email.com', '555-0103',
         '789 Pine Rd', 'Chicago', 'IL', 'USA', '60601', '2022-09-01', 3.2)
    ]
    
    print("\n" + "="*50)
    print("ADDING SAMPLE STUDENTS")
    print("="*50)
    
    for student in students_data:
        if validate_email(student[3]):
            db.add_student(student)
        else:
            print(f"❌ Invalid email for student: {student[3]}")
    
    # Display all students
    print("\n" + "="*50)
    print("ALL STUDENTS")
    print("="*50)
    
    students = db.get_all_students()
    for student in students:
        print(f"ID: {student['student_id']} | "
              f"Name: {student['first_name']} {student['last_name']} | "
              f"Email: {student['email']} | "
              f"Status: {student['student_status']} | "
              f"GPA: {student['gpa']}")
    
    # Get student by ID
    print("\n" + "="*50)
    print("STUDENT DETAILS (ID: 1)")
    print("="*50)
    
    student = db.get_student_by_id(1)
    if student:
        for key, value in student.items():
            print(f"{key}: {value}")
    
    # Search students
    print("\n" + "="*50)
    print("SEARCH RESULTS (term: 'john')")
    print("="*50)
    
    search_results = db.search_students("john")
    for result in search_results:
        print(f"Found: {result['first_name']} {result['last_name']} - {result['email']}")
    
    # Update GPA
    print("\n" + "="*50)
    print("UPDATING GPA")
    print("="*50)
    db.update_gpa(1, 3.75)
    
    # Get statistics
    print("\n" + "="*50)
    print("STUDENT STATISTICS")
    print("="*50)
    
    stats = db.get_statistics()
    if stats:
        print(f"Total Students: {stats['total_students']}")
        print(f"Active Students: {stats['active_students']}")
        print(f"Graduated: {stats['graduated_students']}")
        print(f"Average GPA: {stats['average_gpa']:.2f}")
        print(f"GPA Range: {stats['min_gpa']} - {stats['max_gpa']}")
    
    # Close connection
    db.close()

if __name__ == "__main__":
    main()

✅ Connected to MySQL database
✅ Table 'students' created successfully!

ADDING SAMPLE STUDENTS
✅ Student John Doe added successfully!
✅ Student Jane Smith added successfully!
✅ Student Mike Johnson added successfully!

ALL STUDENTS
ID: 1 | Name: John Doe | Email: john.doe@email.com | Status: active | GPA: 3.50
ID: 3 | Name: Mike Johnson | Email: mike.j@email.com | Status: active | GPA: 3.20
ID: 2 | Name: Jane Smith | Email: jane.smith@email.com | Status: active | GPA: 3.80

STUDENT DETAILS (ID: 1)
student_id: 1
first_name: John
last_name: Doe
date_of_birth: 2000-05-15
email: john.doe@email.com
phone: 555-0101
address: 123 Main St
city: New York
state: NY
country: USA
postal_code: 10001
enrollment_date: 2023-09-01
student_status: active
gpa: 3.50
created_at: 2026-02-26 23:28:48
updated_at: 2026-02-26 23:28:48

SEARCH RESULTS (term: 'john')
Found: John Doe - john.doe@email.com
Found: Mike Johnson - mike.j@email.com

UPDATING GPA
✅ GPA updated for student ID 1

STUDENT STATISTICS
Total St